In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_train, load_test, remove_outliers, split_xy
from src.cleaning import clean

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


In [2]:
train_df = remove_outliers(load_train())
print(len(load_train()), "->", len(train_df), "rows after removing outliers") # 1460 -> 1458 on the real data

X, y = split_xy(train_df)
cleaned = clean(X)

before = X.isna().sum()
after = cleaned.isna().sum()
gaps = pd.DataFrame({"before": before, "after": after.reindex(before.index)})
gaps[gaps["before"] > 0].sort_values("before", ascending=False).head(15)

1460 -> 1458 rows after removing outliers


,before,after
PoolQC,1452,0.0
MiscFeature,1404,0.0
Alley,1367,0.0
Fence,1177,0.0
MasVnrType,872,0.0
FireplaceQu,690,0.0
LotFrontage,259,259.0
GarageType,81,0.0
GarageYrBlt,81,NaN
GarageFinish,81,0.0


In [5]:
from src.features import add_features

fe = add_features(cleaned)
new_cols = [c for c in fe.columns if c not in cleaned.columns]
print(new_cols)

check = fe[new_cols + ["OverallQual", "GrLivArea"]].assign(LogPrice=y.values)
check.corr()["LogPrice"].sort_values(ascending=False).round(2)


['TotalSF', 'TotalBath', 'TotalPorchSF', 'HouseAge', 'RemodAge', 'IsRemodeled', 'IsNew', 'HasPool', 'HasGarage', 'HasBasement', 'HasFireplace', 'Has2ndFloor', 'QualSF']


LogPrice        1.00
QualSF          0.88
TotalSF         0.83
OverallQual     0.82
GrLivArea       0.73
TotalBath       0.68
HasFireplace    0.51
TotalPorchSF    0.40
HasGarage       0.32
IsNew           0.22
HasBasement     0.20
Has2ndFloor     0.15
HasPool         0.08
IsRemodeled    -0.07
RemodAge       -0.57
HouseAge       -0.59
Name: LogPrice, dtype: float64

In [6]:
from sklearn.linear_model import Ridge
from src.pipeline import build_pipeline

pipe = build_pipeline(Ridge(alpha=10)).fit(X, y)
Z = pipe[:-1].transform(X) # everything except the final model
print(Z.shape, "NaN left:", int(np.isnan(Z).sum()))

names = pipe.named_steps["prep"].get_feature_names_out()
print(len(names), "columns, e.g.", list(names[:4]), "...", list(names[-2:]))

skewed = pipe.named_steps["prep"].named_transformers_["num"].named_steps["skew"].skewed_
print(len(skewed), "numeric columns got a log transform")

(1458, 224) NaN left: 0
224 columns, e.g. ['num__LotFrontage', 'num__LotArea', 'num__LotShape', 'num__LandSlope'] ... ['cat__SaleCondition_Partial', 'cat__SaleCondition_infrequent_sklearn']
36 numeric columns got a log transform
